# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [9]:
# import the requests library (1 line)
import requests
from requests import Session # to be implemented

# Create a scraper that builds a JSON file with the political leaders 
# of each country you get from this API: [https://country-leaders.onrender.com/docs]

# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com" 

# assign the /status endpoint to another variable called status_url (1 line)
status_url = "status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(f"{root_url}/{status_url}") # type int

# check the status_code using a condition and print appropriate messages (4 lines)
# print(type(req.status_code))
if str(req.status_code)[0] > "2":
    print(f'Server is not online: status {req.status_code}')
else:
    print(f'Server is online: status {req.status_code}')

Server is online: status 200


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [ ]:
# Set the countries_url variable (1 line)
countries_url = "countries"

# query the /countries endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(f"{root_url}/{countries_url}")

# Get the JSON content and store it in the countries variable (1 line)
countries = req.json() # type dict

# display the request's status code and the countries variable (1 line)
print(req.status_code, countries)
# print(type(req.status_code))

403 {'message': 'The cookie is missing'}
<class 'int'>


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [11]:
# Set the cookie_url variable (1 line)
cookie_url = "cookie"

# Query the endpoint, set the cookies variable and display it (2 lines)
cookies = requests.get(f"{root_url}/{cookie_url}").cookies
print(cookies)



<RequestsCookieJar[<Cookie user_cookie=c80a8630-54ac-49d3-b357-fdaa8342cfbf for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [12]:
# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(f"{root_url}/{countries_url}", cookies=cookies).json()

# display the countries variable (1 line)
print(countries) # type list
# print(countries.request.headers)

['fr', 'us', 'be', 'ma', 'ru']


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [13]:
# Set the leaders_url variable (1 line)
leaders_url = "leaders"

# query the /leaders endpoint, assign the output to the leaders variable (1 line)
leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookies)
# leaders = {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies) for country in countries}
# leaders_per_country = {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":country}).json() for country in countries}

# display the leaders variable (1 line)
print(leaders)

<Response [404]>


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [14]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":"be"})
# {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":"be"}) for country in countries}

# assign the output to the leaders variable (1 line)
leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":"be"}).json()
# display the leaders variable (1 line)
print(leaders)


[{'id': 'Q12978', 'first_name': 'Guy', 'last_name': 'Verhofstadt', 'birth_date': '1953-04-11', 'death_date': None, 'place_of_birth': 'Dendermonde', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Guy_Verhofstadt', 'start_mandate': '1999-07-12', 'end_mandate': '2008-03-20'}, {'id': 'Q12981', 'first_name': 'Yves', 'last_name': 'Leterme', 'birth_date': '1960-10-06', 'death_date': None, 'place_of_birth': 'Wervik', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Yves_Leterme', 'start_mandate': '2009-11-25', 'end_mandate': '2011-12-06'}, {'id': 'Q12983', 'first_name': 'Herman', 'last_name': 'None', 'birth_date': '1947-10-31', 'death_date': None, 'place_of_birth': 'Etterbeek', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Herman_Van_Rompuy', 'start_mandate': '2008-12-30', 'end_mandate': '2009-11-25'}, {'id': 'Q14989', 'first_name': 'Léon', 'last_name': 'Delacroix', 'birth_date': '1867-12-27', 'death_date': '1929-10-15', 'place_of_birth': 'Saint-Josse-ten-Noode', 'wikipedia_url': 'https://nl

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [15]:
# 4 lines

# leaders_per_country = {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":country}).json() for country in countries}
leaders_per_country = {}
for country in countries:
    leaders_info = requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":country}).json()
    leaders_per_country[country] = leaders_info
print(leaders_per_country)
# is a dict in which keys are countries and values dict containing info about leader

# it could have been done in just one (long though) line at first cell of 1d
# with a dict comprehension
# leaders_per_country = {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":country}).json() for country in countries}


{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

In [16]:
# or 1 line
leaders_per_country = {country: requests.get(f"{root_url}/{leaders_url}", cookies=cookies, params={"country":country}).json() for country in countries}
# it's a dict in which keys are countries and values are lists (.json())
    # in which each element is a dict (one for each leader)
        # in which keys are types of personal info and values real info about the leader
print(leaders_per_country)

{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [17]:
# < 15 lines

def get_leaders():
    """
    returns the dict obtained before
    """
    # declaration of url variables
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    countries_url = f'{root_url}/countries'
    leaders_url = f'{root_url}/leaders'
    # getting the cookies
    cookies = requests.get(cookie_url).cookies
    # getting the countries json list
    countries = requests.get(countries_url, cookies=cookies).json()
    leaders_per_country = {}
    for country in countries:
        leaders_info = requests.get(leaders_url, cookies=cookies, params={"country":country}).json()
        leaders_per_country[country] = leaders_info
    return leaders_per_country


In [6]:
# def get_leaders_first_names(country):
#     """
#     Returns first name and last name of all leaders of entered country
#     """
#     leaders_names = []
#     for leader in get_leaders()[country]: # leader is a dict with info about one leader of the country
#         leaders_names.append(f'{leader["first_name"]} {leader["last_name"]}')
#     return leaders_names

# def which_leaders_for_wich_country():
#     leaders_of_countries = {}
#     for key in get_leaders().keys():
#         leaders_of_countries[key] = get_leaders_first_names(key)
#     return leaders_of_countries


# "wikipedia_url" key

# def get_leader_wiki_url(first_name,last_name):
#     if f'{first_name} {last_name}' in which_leaders_for_wich_country():
#         country_of_leader = 

Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [18]:
# 2 lines

leaders_per_country = get_leaders()
# print(leaders_per_country)
print(leaders_per_country)
# print(*leaders_per_country.values(), sep="\n")



{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

In [ ]:
# prints of other functions i made above 

# get_leaders_first_names("fr")
# print(which_leaders_for_wich_country())

## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [7]:
# 3 lines
import requests

# wiki_url = leaders_per_country["fr"][0]["wikipedia_url"]
def get_text(url):
    headers = {"User-Agent":"Wikipedia Scraper Project (https://github.com/ireneghioni-glitch/wikipedia-scraper/tree/main)"}
    # r = requests.get(wiki_url, timeout=10)
    r = requests.get(url, headers=headers, timeout=10)
    return r.text

wiki_url = "https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"
print(get_text(wiki_url))

# without headers I obtained output 
# "Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119"

# code is fine but I'm refused by the server beacuse of too many attempts (Wikimedia Error)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="it" dir="ltr">
<head>
<meta charset="UTF-8">
<title>François Hollande - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled ve

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [8]:
# 3 lines
from bs4 import BeautifulSoup

#loading of html tree of web page
soup = BeautifulSoup(get_text(wiki_url), "html.parser")

print(soup.prettify())


<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="it">
 <head>
  <meta charset="utf-8"/>
  <title>
   François Hollande - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [9]:
# 2 lines
# import selenium 
# from selenium import webdriver
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.common.by import By

paragraphs = [tag.text for tag in soup.find_all("p")]
print(paragraphs)

# print(get_paragraphs()[0]) # prints first paragraph in paragraphs list

# also find might be enough for this task
# print(soup.find("p").text)


['François Gérard Georges Nicolas Hollande (AFI: fʁɑ̃swa ʒeʁaʁ ʒɔʁʒ(ə) nikɔla ɔlɑ̃d · ascoltaⓘ; Rouen, 12 agosto 1954) è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.', "Ex-magistrato presso la Corte dei conti e brevemente avvocato, è membro dell'Assemblea nazionale dal 2024, dopo esserlo già stato due volte (dal 1988 al 1993 e dal 1997 al 2012). Dopo la nomina di Lionel Jospin a Matignon, è stato il primo segretario del Partito Socialista (PS), dal 1997 al 2008, durante la terza coabitazione e poi nell'opposizione. A livello locale, è stato sindaco di Tulle, dal 2001 al 2008, e ha presieduto il consiglio generale di Corrèze, dal 2008 al 2012. È stato altresì Europarlamentare per un breve periodo nel 1999.", "Nominato candidato del PS alle elezioni presidenziali del 2012 dopo le primarie, viene eletto capo dello Stato contro il presidente uscente, Nicolas Sarkozy, con il 51,6% dei voti esp

If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [10]:
# <10 lines
# every biography page starts with the name of the leader
# which is also the title
def get_first_p(url, first_name, last_name):
    soup = BeautifulSoup(get_text(url), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if first_name in paragraph and last_name in paragraph:
            first_paragraph = paragraph
            return first_paragraph



At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [11]:
# 10 lines
# def get_first_paragraph(wikipedia_url):
#   print(wikipedia_url) # keep this for the rest of the notebook
#   [insert your code]
#   return first_paragraph

# a wikipedia url is structured like this
# "https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"
# very last word, preceded by "_", is the last_name of the leader
import re
from urllib.parse import unquote

def get_first_paragraph(wikipedia_url):
    print (wikipedia_url) # keep this for the rest of the notebook
    leader_last_name = unquote(wikipedia_url.split("_")[-1]) # makes a list out of strings in the url seperated on "_" char
    print(leader_last_name)
    soup = BeautifulSoup(get_text(wikipedia_url), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if leader_last_name in paragraph:
            first_paragraph = paragraph
            return first_paragraph


In [60]:
# Test: 3 lines

print(get_first_paragraph("https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"))

https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
Hollande
François Gérard Georges Nicolas Hollande (AFI: fʁɑ̃swa ʒeʁaʁ ʒɔʁʒ(ə) nikɔla ɔlɑ̃d · ascoltaⓘ; Rouen, 12 agosto 1954) è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.


### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [62]:
# 3 lines
#sanitizing with Regex
pattern = "\[\d+\]"
re.sub(pattern, "", get_first_paragraph(wiki_url))

<>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<>:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
C:\Users\irene\AppData\Local\Temp\ipykernel_1196\1276627770.py:3: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
  pattern = "\[\d+\]"


https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
Hollande


'François Gérard Georges Nicolas Hollande (AFI: fʁɑ̃swa ʒeʁaʁ ʒɔʁʒ(ə) nikɔla ɔlɑ̃d · ascoltaⓘ; Rouen, 12 agosto 1954) è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.'

Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [ ]:
# 10 lines
# def get_first_paragraph(wikipedia_url):
#     print (wikipedia_url) # keep this for the rest of the notebook
#     leader_last_name = wikipedia_url.split("_")[-1] # makes a list out of strings in the url seperated on "_" char
#     print(leader_last_name)
#     soup = BeautifulSoup(get_text(wikipedia_url), "html.parser")
#     paragraphs = [tag.text for tag in soup.find_all("p")]
#     for paragraph in paragraphs:
#         if leader_last_name in paragraph[:50]:
#             #sanitizing output with Regex
#             pattern_1 = r"\[\d+\]" # "|" stands for OR in regex
#             pattern_2 = r"\([^()]*\)"
#             while re.search(pattern_2, paragraph):
#                 paragraph = re.sub(pattern_2, "", paragraph)
#             return re.sub(pattern_1, "", paragraph)

# print(get_first_paragraph("https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"))

https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
Hollande
François Gérard Georges Nicolas Hollande  è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.


Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [12]:
# < 20 lines
def get_first_paragraph(wikipedia_url):
    """
    the function returns first cleaned paragraph of the url leader.
    this version of the function is optimized for whatever the url
    """
    print (wikipedia_url) # keep this for the rest of the notebook
    leader_last_name = unquote(wikipedia_url.split("_")[-1]) # makes a list out of strings in the url seperated on "_" char
    soup = BeautifulSoup(get_text(wikipedia_url), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if leader_last_name in paragraph[:50]:
            #sanitizing output with Regex
            pattern_1 = r"\[\d+\]" # "|" stands for OR in regex
            pattern_2 = r"\([^()]*\)"
            while re.search(pattern_2, paragraph):
                paragraph = re.sub(pattern_2, "", paragraph)
            cleaned_paragraph = re.sub(pattern_1, "", paragraph)
            other_patterns = [r"\s+", r"\s+,", r"\s+\.", r"[ⓘ·]"]
            for pattern in other_patterns:
                cleaned_paragraph = re.sub(pattern, " ", cleaned_paragraph)
            return cleaned_paragraph

print(get_first_paragraph("https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"))
# output:
# François Gérard Georges Nicolas Hollande è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.

print(get_first_paragraph("https://it.wikipedia.org/wiki/Giorgia_Meloni"))
# output:
# Giorgia Meloni è una politica italiana, presidente del Consiglio dei ministri della Repubblica Italiana dal 22 ottobre 2022. Prima donna a capo del governo nella storia d'Italia, ha ricoperto precedentemente gli incarichi di vicepresidente della Camera dei deputati dal 2006 al 2008 e di ministro per la gioventù nel quarto governo Berlusconi dal 2008 al 2011.

print(get_first_paragraph("https://it.wikipedia.org/wiki/Pedro_S%C3%A1nchez"))

https://it.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Gérard Georges Nicolas Hollande è un funzionario e politico francese, Presidente della Repubblica francese dal 2012 al 2017 e segretario del Partito Socialista dal 1997 al 2008.
https://it.wikipedia.org/wiki/Giorgia_Meloni
Giorgia Meloni è una politica italiana, presidente del Consiglio dei ministri della Repubblica Italiana dal 22 ottobre 2022. Prima donna a capo del governo nella storia d'Italia, ha ricoperto precedentemente gli incarichi di vicepresidente della Camera dei deputati dal 2006 al 2008 e di ministro per la gioventù nel quarto governo Berlusconi dal 2008 al 2011. 
https://it.wikipedia.org/wiki/Pedro_S%C3%A1nchez
Pedro Sánchez Pérez-Castejón è un politico ed economista spagnolo, presidente del Governo di Spagna a partire dal 2 giugno 2018.


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
# < 20 lines\def get_leaders():
# this function has been dismissed in the last version of get_leaders(), see end of notebook
def manage_cookies_expiring(country):
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    leaders_url = f'{root_url}/leaders'
    cookies = requests.get(cookie_url).cookies
    response = requests.get(leaders_url, cookies=cookies, params={"country":country})
    return response

def get_leaders():
    """
    New version of get_leaders() function.
    Now it loops over each leader to extract the leader url then
    Executes get_first_paragraph() just optimized then
    Updates dict named leader with leader name (key) and first cleaned paragraph (value)
    untill dictionary is complete.
    Returns the leader dictionary.
    """
    # declaration of url variables
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    countries_url = f'{root_url}/countries'
    leaders_url = f'{root_url}/leaders'
    # getting the cookies
    cookies = requests.get(cookie_url).cookies
    # getting the countries json list
    countries = requests.get(countries_url, cookies=cookies).json()
    leaders_per_country = {}
    for country in countries:
        response = requests.get(leaders_url, cookies=cookies, params={"country":country})
        if int(str(response.status_code)[0]) > 2:
            print(f'Cookies for {country} expired! They will now be restored...')
            response = manage_cookies_expiring(country)
        leaders_info = response.json()
        leaders_per_country[country] = leaders_info
        for leader in leaders_info:
            # print(leader) # prints a dictionary for each leader as expected
            # print(type(leader))
            leader_intro = get_first_paragraph(unquote(leader["wikipedia_url"]))
            leader[f'{leader["first_name"]} {leader["last_name"]} intro'] = leader_intro
    return leaders_per_country  
    # for country, group_of_country_leaders in leaders_per_country.items():
    #     for leader in group_of_country_leaders:
    #         leader_intro = get_first_paragraph(unquote(leader["wikipedia_url"]))
    #         leader[f'{leader["first_name"]} {leader["last_name']} intro'] = leader_intro
    # return leaders_per_country

In [83]:
# Check the output of your function (2 lines)
print(get_leaders())

<class 'dict'>
Paese: fr | Tipo: <class 'dict'> | Contenuto: {'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}
https://fr.wikipedia.org/wiki/François_Hollande
<class 'dict'>
Paese: fr | Tipo: <class 'dict'> | Contenuto: {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
<class 'dict'>
Paese: fr | Tipo: <class 'dict'> | Contenuto: {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wi

TypeError: string indices must be integers, not 'str'

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
# < 25 lines
try:
    print(get_leaders())
except:
    print('get_leaders() function failed, not because of cookies expiring.')


https://fr.wikipedia.org/wiki/François_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/François_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Valéry_Giscard_d'Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napoléon_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/René_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/Émile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincaré
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d'État)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.org/wiki/Félix_Faure
https://fr.wikipedia.or

Check the output of your function again.

In [ ]:
# Check the output of your function (1 line)
print(get_leaders())

# took 2m and 33sec

https://fr.wikipedia.org/wiki/François_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/François_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Valéry_Giscard_d'Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napoléon_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/René_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/Émile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincaré
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d'État)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.org/wiki/Félix_Faure
https://fr.wikipedia.or

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [ ]:
# < 20 lines
from requests import Session

# func below is not used, keeping it here for the moment
# def session_request(response, country):
#     leaders_info = response.json()
#     leaders_per_country[country] = leaders_info
#     return leaders_info

# # this will be the orchestra director (main())
# def session_get_leaders_request():
#     # session definition
#     with Session() as session:
#         # here in the main() of main.py there will be 2 variables
#         # outpout of API_client
#         # which wiil be input of scraping part
#         # now get_leaders does both
#         return get_leaders(session)

def get_text(url:str, session:Session):
    headers = {"User-Agent":"Wikipedia Scraper Project (https://github.com/ireneghioni-glitch/wikipedia-scraper/tree/main)"}
    # r = requests.get(wiki_url, timeout=10)
    r = session.get(url, headers=headers, timeout=10)
    return r.text

def get_first_paragraph(wikipedia_url, session:Session):
    """
    the function returns first cleaned paragraph of the url leader.
    this version of the function is optimized for whatever the url
    """
    print (wikipedia_url) # keep this for the rest of the notebook
    leader_last_name = unquote(wikipedia_url.split("_")[-1]) # makes a list out of strings in the url seperated on "_" char
    soup = BeautifulSoup(get_text(wikipedia_url, session), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if leader_last_name in paragraph[:50]:
            #sanitizing output with Regex
            pattern_1 = r"\[\d+\]" # "|" stands for OR in regex
            pattern_2 = r"\([^()]*\)"
            while re.search(pattern_2, paragraph):
                paragraph = re.sub(pattern_2, "", paragraph)
            cleaned_paragraph = re.sub(pattern_1, "", paragraph)
            other_patterns = [r"\s+", r"\s+,", r"\s+\.", r"[ⓘ·]"]
            for pattern in other_patterns:
                cleaned_paragraph = re.sub(pattern, " ", cleaned_paragraph)
            return cleaned_paragraph

def get_leaders(session):
    """
    New version of get_leaders() function.
    Now it receives session.
    It loops over each leader to extract the leader url then
    Executes get_first_paragraph() that has just been optimized then
    Adds to dict of leader the leader name (key) and first cleaned paragraph (value)
    untill dictionary is complete.
    Returns the leader dictionary.
    """
    # declaration of url variables
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    countries_url = f'{root_url}/countries'
    leaders_url = f'{root_url}/leaders'
    # getting the cookies
    cookies = session.get(cookie_url).cookies
    # getting the countries json list
    countries = session.get(countries_url, cookies=cookies).json()
    leaders_per_country = {}
    for country in countries:
        response = session.get(leaders_url, params={"country":country}) # change with session
        if int(str(response.status_code)[0]) > 2:
            print(f'Cookies for {country} expired! Restoring...')
            session.get(cookie_url) # updating internal cookie status
            response = session.get(leaders_url, params={"country":country}) # new cookie status passed through session
        leaders_info = response.json()
        leaders_per_country[country] = leaders_info
        for leader in leaders_info:
            # print(leader) # prints a dictionary for each leader as expected
            # print(type(leader))
            leader_intro = get_first_paragraph(leader["wikipedia_url"], session)
            leader["first_paragraph"] = leader_intro
    return leaders_per_country

print(session_get_leaders_request())

# took 1m and 1sec

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [ ]:
# <25 lines

# ups made it above

Test your new functions.



In [ ]:
# see above

# print(session_get_leaders_request())

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [ ]:
# 3 lines

# saveing output will be a responsability of the orchestra director 
# (which is session_get_leaders_request() in this notebook)

import json

# this will be the orchestra director (main())
def session_get_leaders_request():
    # session definition
    with Session() as session:
        # here in the main() of main.py there will be 2 variables
        # outpout of API_client
        # which wiil be input of scraping part
        # now get_leaders does both
        leaders_per_country = get_leaders(session)
    
    with open("leaders.json", "w", encoding="utf-8") as f:
        json.dump(leaders_per_country, f, indent=4, ensure_ascii=False)
    
    return leaders_per_country

def get_text(url:str, session:Session):
    headers = {"User-Agent":"Wikipedia Scraper Project (https://github.com/ireneghioni-glitch/wikipedia-scraper/tree/main)"}
    # r = requests.get(wiki_url, timeout=10)
    r = session.get(url, headers=headers, timeout=10)
    return r.text

def get_first_paragraph(wikipedia_url, session:Session):
    """
    the function returns first cleaned paragraph of the url leader.
    this version of the function is optimized for whatever the url
    """
    print (wikipedia_url) # keep this for the rest of the notebook
    leader_last_name = unquote(wikipedia_url.split("_")[-1]) # makes a list out of strings in the url seperated on "_" char
    soup = BeautifulSoup(get_text(wikipedia_url, session), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if leader_last_name in paragraph[:50]:
            #sanitizing output with Regex
            pattern_1 = r"\[\d+\]" # "|" stands for OR in regex
            pattern_2 = r"\([^()]*\)"
            while re.search(pattern_2, paragraph):
                paragraph = re.sub(pattern_2, "", paragraph)
            cleaned_paragraph = re.sub(pattern_1, "", paragraph)
            other_patterns = [r"\s+", r"\s+,", r"\s+\.", r"[ⓘ·]"]
            for pattern in other_patterns:
                cleaned_paragraph = re.sub(pattern, " ", cleaned_paragraph)
            return cleaned_paragraph

def get_leaders(session):
    """
    New version of get_leaders() function.
    Now it receives session.
    It loops over each leader to extract the leader url then
    Executes get_first_paragraph() that has just been optimized then
    Adds to dict of leader the leader name (key) and first cleaned paragraph (value)
    untill dictionary is complete.
    Returns the leader dictionary.
    """
    # declaration of url variables
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    countries_url = f'{root_url}/countries'
    leaders_url = f'{root_url}/leaders'
    # getting the cookies
    cookies = session.get(cookie_url).cookies
    # getting the countries json list
    countries = session.get(countries_url, cookies=cookies).json()
    leaders_per_country = {}
    for country in countries:
        response = session.get(leaders_url, params={"country":country}) # change with session
        if int(str(response.status_code)[0]) > 2:
            print(f'Cookies for {country} expired! Restoring...')
            session.get(cookie_url) # updating internal cookie status
            response = session.get(leaders_url, params={"country":country}) # new cookie status passed through session
        leaders_info = response.json()
        leaders_per_country[country] = leaders_info
        for leader in leaders_info:
            # print(leader) # prints a dictionary for each leader as expected
            # print(type(leader))
            leader_intro = get_first_paragraph(leader["wikipedia_url"], session)
            leader["first_paragraph"] = leader_intro
    return leaders_per_country

print(session_get_leaders_request())


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

In [ ]:
# check for leaders.json location
import os
print(os.getcwd())

d:\Irene\Desktop\AI & Data Science training - BeCode\wikipedia-scraper\dev


Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [25]:
# 3 lines
with open("leaders.json", "r", encoding="utf-8") as f:
    print(json.load(f))

{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14', 'first_paragraph': 'Pour les articles homonymes, voir Hollande  '}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15', 'first_paragraph': '« Sarkozy » redirige ici. Pour les autres significations, voir Sarkozy  Nagy et Bocșa. '}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': 

Make a function `save(leaders_per_country)` to call this code easily.

In [ ]:
# 3 lines
# written here but implemented in session_get_leaders_request() in the cell below
def save(leaders_per_country):
    with open("leaders.json", "w", encoding="utf-8") as f:
        return json.dump(leaders_per_country, f, indent=4, ensure_ascii=False)


In [26]:
# Call the function (1 line)

# saveing output will be a responsability of the orchestra director 
# (which is session_get_leaders_request() in this notebook)

import json

# this will be the orchestra director (main())
def session_get_leaders_request():
    # session definition
    with Session() as session:
        # here in the main() of main.py there will be 2 variables
        # outpout of API_client
        # which wiil be input of scraping part
        # now get_leaders does both
        leaders_per_country = get_leaders(session)
    
    save(leaders_per_country)
    
    return leaders_per_country

def save(leaders_per_country):
    with open("leaders.json", "w", encoding="utf-8") as f:
        return json.dump(leaders_per_country, f, indent=4, ensure_ascii=False)

def get_text(url:str, session:Session):
    headers = {"User-Agent":"Wikipedia Scraper Project (https://github.com/ireneghioni-glitch/wikipedia-scraper/tree/main)"}
    # r = requests.get(wiki_url, timeout=10)
    r = session.get(url, headers=headers, timeout=10)
    return r.text

def get_first_paragraph(wikipedia_url, session:Session):
    """
    the function returns first cleaned paragraph of the url leader.
    this version of the function is optimized for whatever the url
    """
    print (wikipedia_url) # keep this for the rest of the notebook
    leader_last_name = unquote(wikipedia_url.split("_")[-1]) # makes a list out of strings in the url seperated on "_" char
    soup = BeautifulSoup(get_text(wikipedia_url, session), "html.parser")
    paragraphs = [tag.text for tag in soup.find_all("p")]
    for paragraph in paragraphs:
        if leader_last_name in paragraph[:50]:
            #sanitizing output with Regex
            pattern_1 = r"\[\d+\]" # "|" stands for OR in regex
            pattern_2 = r"\([^()]*\)"
            while re.search(pattern_2, paragraph):
                paragraph = re.sub(pattern_2, "", paragraph)
            cleaned_paragraph = re.sub(pattern_1, "", paragraph)
            other_patterns = [r"\s+", r"\s+,", r"\s+\.", r"[ⓘ·]"]
            for pattern in other_patterns:
                cleaned_paragraph = re.sub(pattern, " ", cleaned_paragraph)
            return cleaned_paragraph

def get_leaders(session):
    """
    New version of get_leaders() function.
    Now it receives session.
    It loops over each leader to extract the leader url then
    Executes get_first_paragraph() that has just been optimized then
    Adds to dict of leader the leader name (key) and first cleaned paragraph (value)
    untill dictionary is complete.
    Returns the leader dictionary.
    """
    # declaration of url variables
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f'{root_url}/cookie'
    countries_url = f'{root_url}/countries'
    leaders_url = f'{root_url}/leaders'
    # getting the cookies
    cookies = session.get(cookie_url).cookies
    # getting the countries json list
    countries = session.get(countries_url, cookies=cookies).json()
    leaders_per_country = {}
    for country in countries:
        response = session.get(leaders_url, params={"country":country}) # change with session
        if int(str(response.status_code)[0]) > 2:
            print(f'Cookies for {country} expired! Restoring...')
            session.get(cookie_url) # updating internal cookie status
            response = session.get(leaders_url, params={"country":country}) # new cookie status passed through session
        leaders_info = response.json()
        leaders_per_country[country] = leaders_info
        for leader in leaders_info:
            # print(leader) # prints a dictionary for each leader as expected
            # print(type(leader))
            leader_intro = get_first_paragraph(leader["wikipedia_url"], session)
            leader["first_paragraph"] = leader_intro
    return leaders_per_country

print(session_get_leaders_request())

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!